In [3]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient


import time 

import json

In [2]:
load_dotenv()
client = MongoClient(os.getenv("MONGO_URI"))   # connexion au cluster Atlas

# "ping" = on demande au serveur s'il répond
reponse = client.admin.command("ping")
print("Réponse d'Atlas :", reponse)

Réponse d'Atlas : {'ok': 1}


## 1. Insertion des films bruts
Base : `cinema` — Collection : `films_bruts` (un document = un film, avec son casting filtré).
L'identifiant TMDB devient la clé MongoDB (`_id`) → chaque film est unique.
Le nettoyage des données sera réalisé à l'étape 2 (Traitement).

In [4]:
# Lecture du fichier : une ligne = un film
with open("../data/films_details.jsonl", encoding="utf-8") as f:
    films = [json.loads(ligne) for ligne in f]

# L'id TMDB devient l'identifiant MongoDB
for film in films:
    film["_id"] = film["id"]

print(len(films), "films prêts à être insérés")

6015 films prêts à être insérés


In [5]:
db = client["cinema"]                 # base de données (créée automatiquement)
collection = db["films_bruts"]        # collection (créée automatiquement)

collection.delete_many({})            # vide la collection : la cellule peut être relancée sans doublons
resultat = collection.insert_many(films)

print("Films insérés :", len(resultat.inserted_ids))
print("Films dans la collection :", collection.count_documents({}))

Films insérés : 6015
Films dans la collection : 6015
